In [3]:
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import statsmodels.api as sm
from statsmodels.formula.api import ols
from glob import glob
from pathlib import Path
from scipy.stats import mannwhitneyu, ttest_ind, ttest_rel, wilcoxon

In [9]:
figure2_hdata=pd.read_csv("Figure/Figure2/PanelH/intensity_data_long.csv")
figure2_hdata.head()

,value,type
0,5.810,early_control_pre
1,9.360,early_control_pre
2,8.110,early_control_pre
3,15.572,early_control_pre
4,13.214,early_control_pre


In [10]:
# Remove existing parsed-factor columns in case this cell is re-run.
figure2_hdata = figure2_hdata.drop(columns=["stage", "treatment", "time"], errors="ignore")

figure2_hdata[["stage", "treatment", "time"]] = figure2_hdata["type"].str.extract(
    r"(?P<stage>early|late)_(?P<treatment>control|cytoD)_(?P<time>pre|post)"
)

for col in ["stage", "treatment", "time"]:
    figure2_hdata[col] = figure2_hdata[col].astype("category")

print(figure2_hdata.head())

    value               type  stage treatment time
0   5.810  early_control_pre  early   control  pre
1   9.360  early_control_pre  early   control  pre
2   8.110  early_control_pre  early   control  pre
3  15.572  early_control_pre  early   control  pre
4  13.214  early_control_pre  early   control  pre


In [12]:
model = ols("value ~ C(stage) * C(treatment) * C(time)", data=figure2_hdata).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

                                    sum_sq     df          F        PR(>F)
C(stage)                         42.831973    1.0   1.820441  1.796803e-01
C(treatment)                    428.054153    1.0  18.193123  3.883936e-05
C(time)                        1730.230724    1.0  73.538126  3.045997e-14
C(stage):C(treatment)           102.593174    1.0   4.360407  3.879718e-02
C(stage):C(time)                  5.757629    1.0   0.244710  6.216860e-01
C(treatment):C(time)              6.078651    1.0   0.258354  6.121418e-01
C(stage):C(treatment):C(time)     3.955109    1.0   0.168100  6.825024e-01
Residual                       2964.572036  126.0        NaN           NaN


In [13]:
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# 1) Pre vs post within each treatment (pooling stages)
rows_treatment = []
for treatment, sub in figure2_hdata.groupby("treatment"):
    pre = sub.loc[sub["time"] == "pre", "value"].dropna()
    post = sub.loc[sub["time"] == "post", "value"].dropna()
    stat, p = ttest_ind(pre, post, equal_var=False)  # Welch's t-test
    rows_treatment.append(
        {
            "treatment": treatment,
            "n_pre": len(pre),
            "n_post": len(post),
            "mean_pre": pre.mean(),
            "mean_post": post.mean(),
            "delta_post_minus_pre": post.mean() - pre.mean(),
            "t_stat": stat,
            "p_value": p,
        }
    )

prepost_by_treatment = pd.DataFrame(rows_treatment)
prepost_by_treatment["p_fdr_bh"] = multipletests(
    prepost_by_treatment["p_value"], method="fdr_bh"
)[1]

print("Pre vs post within each treatment (pooled over stage)")
display(prepost_by_treatment.sort_values("p_value"))

# 2) Pre vs post within each stage x treatment group
rows_stage_treatment = []
for (stage, treatment), sub in figure2_hdata.groupby(["stage", "treatment"]):
    pre = sub.loc[sub["time"] == "pre", "value"].dropna()
    post = sub.loc[sub["time"] == "post", "value"].dropna()
    stat, p = ttest_ind(pre, post, equal_var=False)
    rows_stage_treatment.append(
        {
            "stage": stage,
            "treatment": treatment,
            "n_pre": len(pre),
            "n_post": len(post),
            "mean_pre": pre.mean(),
            "mean_post": post.mean(),
            "delta_post_minus_pre": post.mean() - pre.mean(),
            "t_stat": stat,
            "p_value": p,
        }
    )

prepost_by_stage_treatment = pd.DataFrame(rows_stage_treatment)
prepost_by_stage_treatment["p_fdr_bh"] = multipletests(
    prepost_by_stage_treatment["p_value"], method="fdr_bh"
)[1]

print("\nPre vs post within each stage x treatment group")
display(prepost_by_stage_treatment.sort_values(["stage", "treatment"]))

Pre vs post within each treatment (pooled over stage)


,treatment,n_pre,n_post,mean_pre,mean_post,delta_post_minus_pre,t_stat,p_value,p_fdr_bh
0,control,54,54,10.081870,17.414000,7.332130,-7.272349,9.173737e-11,1.834747e-10
1,cytoD,13,13,6.163692,12.746308,6.582615,-5.667248,1.376315e-05,1.376315e-05



Pre vs post within each stage x treatment group


,stage,treatment,n_pre,n_post,mean_pre,mean_post,delta_post_minus_pre,t_stat,p_value,p_fdr_bh
0,early,control,14,14,11.288643,19.611214,8.322571,-5.395214,2.464464e-05,0.000049
1,early,cytoD,9,9,5.467000,11.875667,6.408667,-4.741697,5.118170e-04,0.000682
2,late,control,40,40,9.659500,16.644975,6.985475,-5.682532,2.548532e-07,0.000001
3,late,cytoD,4,4,7.731250,14.705250,6.974000,-3.702644,1.243706e-02,0.012437


In [14]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Tukey HSD pre vs post within each treatment
# Output includes mean difference, adjusted p-value, CI, and reject decision.
tukey_rows = []
for treatment, sub in figure2_hdata.groupby("treatment"):
    sub = sub.dropna(subset=["value", "time"])
    if sub["time"].nunique() < 2:
        continue

    tukey = pairwise_tukeyhsd(endog=sub["value"], groups=sub["time"], alpha=0.05)
    tdf = pd.DataFrame(tukey._results_table.data[1:], columns=tukey._results_table.data[0])
    tdf["treatment"] = treatment
    tukey_rows.append(tdf)

if tukey_rows:
    tukey_prepost_by_treatment = pd.concat(tukey_rows, ignore_index=True)
    cols = ["treatment", "group1", "group2", "meandiff", "p-adj", "lower", "upper", "reject"]
    tukey_prepost_by_treatment = tukey_prepost_by_treatment[cols]
    display(tukey_prepost_by_treatment)
else:
    print("No valid treatment groups found for Tukey HSD.")

,treatment,group1,group2,meandiff,p-adj,lower,upper,reject
0,control,post,pre,-7.3321,0.0,-9.3310,-5.3332,True
1,cytoD,post,pre,-6.5826,0.0,-8.9799,-4.1854,True


In [15]:
# Tukey HSD pre vs post within each stage x treatment subgroup
# This gives multiple-comparison-adjusted p-values and CIs for each subgroup.

tukey_rows_stage_treatment = []
for (stage, treatment), sub in figure2_hdata.groupby(["stage", "treatment"]):
    sub = sub.dropna(subset=["value", "time"])
    if sub["time"].nunique() < 2:
        continue

    tukey = pairwise_tukeyhsd(endog=sub["value"], groups=sub["time"], alpha=0.05)
    tdf = pd.DataFrame(tukey._results_table.data[1:], columns=tukey._results_table.data[0])
    tdf["stage"] = stage
    tdf["treatment"] = treatment
    tukey_rows_stage_treatment.append(tdf)

if tukey_rows_stage_treatment:
    tukey_prepost_by_stage_treatment = pd.concat(tukey_rows_stage_treatment, ignore_index=True)
    cols = ["stage", "treatment", "group1", "group2", "meandiff", "p-adj", "lower", "upper", "reject"]
    tukey_prepost_by_stage_treatment = tukey_prepost_by_stage_treatment[cols]
    display(tukey_prepost_by_stage_treatment.sort_values(["stage", "treatment"]))
else:
    print("No valid stage x treatment subgroups found for Tukey HSD.")

,stage,treatment,group1,group2,meandiff,p-adj,lower,upper,reject
0,early,control,post,pre,-8.3226,0.0000,-11.4934,-5.1517,True
1,early,cytoD,post,pre,-6.4087,0.0002,-9.2738,-3.5435,True
2,late,control,post,pre,-6.9855,0.0000,-9.4328,-4.5381,True
3,late,cytoD,post,pre,-6.9740,0.0101,-11.5828,-2.3652,True


In [17]:


# Figure 4B: control vs keratin morphant angle-bin statistics
# Uses per-embryo proportions so embryos with more cells do not dominate.

def load_angles(file_path):
    angles = pd.read_csv(file_path)["Angle"].to_numpy(dtype=float)
    angles = np.where(angles > 90, 180 - angles, angles)
    angles = np.where(angles < 0, angles + 180, angles)
    return angles

control_files = sorted(glob("data/Fig4/4B/*Control*.csv"))
morphant_files = sorted(glob("data/Fig4/4B/*K4K8MO*.csv"))

print(f"Control embryos: {len(control_files)}")
print(f"Keratin morphant embryos: {len(morphant_files)}")

angle_bins = np.linspace(0, 90, 10)  # 0,10,...,90

rows = []
for group, files in [("control", control_files), ("morphant", morphant_files)]:
    for fp in files:
        angles = load_angles(fp)
        hist, edges = np.histogram(angles, bins=angle_bins)
        total = hist.sum()
        props = hist / total if total > 0 else np.zeros_like(hist, dtype=float)

        embryo_id = Path(fp).stem
        for i in range(len(hist)):
            rows.append(
                {
                    "group": group,
                    "embryo": embryo_id,
                    "bin_low": float(edges[i]),
                    "bin_high": float(edges[i + 1]),
                    "count": int(hist[i]),
                    "prop": float(props[i]),
                }
            )

bin_df = pd.DataFrame(rows)

def per_bin_tests(df):
    out = []
    for (lo, hi), sub in df.groupby(["bin_low", "bin_high"], sort=True):
        ctrl = sub.loc[sub["group"] == "control", "prop"].to_numpy()
        mo = sub.loc[sub["group"] == "morphant", "prop"].to_numpy()

        if len(ctrl) == 0 or len(mo) == 0:
            continue

        t_stat, p_t = ttest_ind(ctrl, mo, equal_var=False)
        u_stat, p_u = mannwhitneyu(ctrl, mo, alternative="two-sided")

        out.append(
            {
                "bin": f"{int(lo)}-{int(hi)}",
                "ctrl_mean_prop": ctrl.mean(),
                "mo_mean_prop": mo.mean(),
                "delta_mo_minus_ctrl": mo.mean() - ctrl.mean(),
                "welch_t_p": p_t,
                "mannwhitney_p": p_u,
                "n_control": len(ctrl),
                "n_morphant": len(mo),
            }
        )
    return pd.DataFrame(out)

bin_stats = per_bin_tests(bin_df)

# FDR correction across bins for each test family
from statsmodels.stats.multitest import multipletests
bin_stats["welch_t_p_fdr"] = multipletests(bin_stats["welch_t_p"], method="fdr_bh")[1]
bin_stats["mannwhitney_p_fdr"] = multipletests(bin_stats["mannwhitney_p"], method="fdr_bh")[1]

print("Per-bin control vs morphant tests (based on per-embryo proportions)")
display(bin_stats)

# Primary question: bins < 30 degrees (0-10, 10-20, 20-30) combined
lt30 = (
    bin_df.loc[bin_df["bin_high"] <= 30]
    .groupby(["group", "embryo"], as_index=False)["prop"]
    .sum()
    .rename(columns={"prop": "prop_lt30"})
)

ctrl_lt30 = lt30.loc[lt30["group"] == "control", "prop_lt30"].to_numpy()
mo_lt30 = lt30.loc[lt30["group"] == "morphant", "prop_lt30"].to_numpy()

t_stat_lt30, p_t_lt30 = ttest_ind(ctrl_lt30, mo_lt30, equal_var=False)
u_stat_lt30, p_u_lt30 = mannwhitneyu(ctrl_lt30, mo_lt30, alternative="two-sided")

lt30_stats = pd.DataFrame(
    [
        {
            "metric": "Proportion in bins <30 degrees",
            "control_mean": ctrl_lt30.mean(),
            "morphant_mean": mo_lt30.mean(),
            "delta_mo_minus_ctrl": mo_lt30.mean() - ctrl_lt30.mean(),
            "welch_t_p": p_t_lt30,
            "mannwhitney_p": p_u_lt30,
            "n_control": len(ctrl_lt30),
            "n_morphant": len(mo_lt30),
        }
    ]
)

print("\nCombined <30 degrees comparison")
display(lt30_stats)

# Optional: save for reporting
out_dir = Path("Figure/Figure4/Stats")
out_dir.mkdir(parents=True, exist_ok=True)
bin_stats.to_csv(out_dir / "Fig4B_per_bin_control_vs_morphant.csv", index=False)
lt30_stats.to_csv(out_dir / "Fig4B_lt30_control_vs_morphant.csv", index=False)
print(f"Saved tables to: {out_dir}")

Control embryos: 3
Keratin morphant embryos: 4
Per-bin control vs morphant tests (based on per-embryo proportions)


,bin,ctrl_mean_prop,mo_mean_prop,delta_mo_minus_ctrl,welch_t_p,mannwhitney_p,n_control,n_morphant,welch_t_p_fdr,mannwhitney_p_fdr
0,0-10,0.031181,0.065215,0.034034,0.107947,0.153576,3,4,0.393202,0.685714
1,10-20,0.037465,0.076058,0.038593,0.131067,0.114286,3,4,0.393202,0.685714
2,20-30,0.045048,0.058017,0.012969,0.604169,0.400000,3,4,0.776123,0.720000
3,30-40,0.072777,0.071256,-0.001521,0.974969,0.628571,3,4,0.974969,0.808163
4,40-50,0.088559,0.100944,0.012385,0.654479,0.628571,3,4,0.776123,0.808163
5,50-60,0.143106,0.122749,-0.020357,0.089501,0.228571,3,4,0.393202,0.685714
6,60-70,0.173124,0.161267,-0.011857,0.689887,0.857143,3,4,0.776123,0.857143
7,70-80,0.194905,0.169917,-0.024988,0.662426,0.857143,3,4,0.776123,0.857143
8,80-90,0.213835,0.174577,-0.039258,0.479991,0.400000,3,4,0.776123,0.720000



Combined <30 degrees comparison


,metric,control_mean,morphant_mean,delta_mo_minus_ctrl,welch_t_p,mannwhitney_p,n_control,n_morphant
0,Proportion in bins <30 degrees,0.113694,0.199289,0.085596,0.18295,0.228571,3,4


Saved tables to: Figure/Figure4/Stats


In [19]:
pipviscosity=pd.read_csv("data/Fig4/4C/K4K8MOViscosity.csv")
pipviscosity.head()

,,Eta,Gamma,Stage,Exp Sample,Unnamed: 5
0,0,2936.081,7153.099,early,Control MO,NaN
1,1,3559.848,6148.773,early,Control MO,NaN
2,2,3603.803,7579.641,early,Control MO,NaN
3,3,3828.540,6689.172,early,Control MO,NaN
4,4,3166.910,8204.421,early,Control MO,NaN


In [20]:
from statsmodels.stats.multitest import multipletests

# Pairwise Welch t-tests: Control MO vs each other Exp Sample
# Runs for Eta and Gamma (overall, and stage-wise).

def control_vs_others_ttests(df, value_col, control_label="Control MO", group_col="Exp Sample", stage_col="Stage"):
    d = df[[value_col, group_col, stage_col]].dropna().copy()
    d[group_col] = d[group_col].astype(str).str.strip()
    d[stage_col] = d[stage_col].astype(str).str.strip()

    groups = sorted(d[group_col].unique())
    if control_label not in groups:
        raise ValueError(f"'{control_label}' not found in {group_col}. Found: {groups}")

    others = [g for g in groups if g != control_label]

    # Overall comparisons
    rows_overall = []
    x = d.loc[d[group_col] == control_label, value_col].to_numpy()
    for g in others:
        y = d.loc[d[group_col] == g, value_col].to_numpy()
        stat, p = ttest_ind(x, y, equal_var=False)
        rows_overall.append(
            {
                "value": value_col,
                "comparison": f"{control_label} vs {g}",
                "stage": "all",
                "n_control": len(x),
                "n_other": len(y),
                "mean_control": np.mean(x),
                "mean_other": np.mean(y),
                "delta_other_minus_control": np.mean(y) - np.mean(x),
                "t_stat": stat,
                "p_value": p,
            }
        )

    overall = pd.DataFrame(rows_overall)
    if not overall.empty:
        overall["p_fdr_bh"] = multipletests(overall["p_value"], method="fdr_bh")[1]

    # Stage-wise comparisons
    rows_stage = []
    for stage, sub in d.groupby(stage_col):
        x = sub.loc[sub[group_col] == control_label, value_col].to_numpy()
        if len(x) < 2:
            continue
        for g in others:
            y = sub.loc[sub[group_col] == g, value_col].to_numpy()
            if len(y) < 2:
                continue
            stat, p = ttest_ind(x, y, equal_var=False)
            rows_stage.append(
                {
                    "value": value_col,
                    "comparison": f"{control_label} vs {g}",
                    "stage": stage,
                    "n_control": len(x),
                    "n_other": len(y),
                    "mean_control": np.mean(x),
                    "mean_other": np.mean(y),
                    "delta_other_minus_control": np.mean(y) - np.mean(x),
                    "t_stat": stat,
                    "p_value": p,
                }
            )

    stagewise = pd.DataFrame(rows_stage)
    if not stagewise.empty:
        stagewise["p_fdr_bh"] = multipletests(stagewise["p_value"], method="fdr_bh")[1]

    return overall, stagewise

# Clean accidental unnamed columns if present
pipvisc = pipviscosity.loc[:, ~pipviscosity.columns.str.contains(r"^Unnamed")].copy()

eta_overall, eta_stagewise = control_vs_others_ttests(pipvisc, value_col="Eta")
gamma_overall, gamma_stagewise = control_vs_others_ttests(pipvisc, value_col="Gamma")

print("Available Exp Sample groups:", sorted(pipvisc["Exp Sample"].astype(str).str.strip().unique()))

print("\nEta: Control MO vs others (overall)")
display(eta_overall.sort_values("p_value"))

print("Eta: Control MO vs others (stage-wise)")
display(eta_stagewise.sort_values(["stage", "p_value"]))

print("\nGamma: Control MO vs others (overall)")
display(gamma_overall.sort_values("p_value"))

print("Gamma: Control MO vs others (stage-wise)")
display(gamma_stagewise.sort_values(["stage", "p_value"]))

Available Exp Sample groups: ['Control MO', 'K4K8MO']

Eta: Control MO vs others (overall)


,value,comparison,stage,n_control,n_other,mean_control,mean_other,delta_other_minus_control,t_stat,p_value,p_fdr_bh
0,Eta,Control MO vs K4K8MO,all,34,26,5063.118506,2625.213261,-2437.905245,4.401528,0.000054,0.000054


Eta: Control MO vs others (stage-wise)


,value,comparison,stage,n_control,n_other,mean_control,mean_other,delta_other_minus_control,t_stat,p_value,p_fdr_bh
0,Eta,Control MO vs K4K8MO,early,21,13,3862.134771,1327.613521,-2534.521251,8.125096,6.524449e-08,1.304890e-07
1,Eta,Control MO vs K4K8MO,late,13,13,7003.169154,3922.813001,-3080.356153,3.636764,1.353600e-03,1.353600e-03



Gamma: Control MO vs others (overall)


,value,comparison,stage,n_control,n_other,mean_control,mean_other,delta_other_minus_control,t_stat,p_value,p_fdr_bh
0,Gamma,Control MO vs K4K8MO,all,34,26,9500.136765,2161.409425,-7338.727339,13.585986,1.227223e-19,1.227223e-19


Gamma: Control MO vs others (stage-wise)


,value,comparison,stage,n_control,n_other,mean_control,mean_other,delta_other_minus_control,t_stat,p_value,p_fdr_bh
0,Gamma,Control MO vs K4K8MO,early,21,13,9997.535143,1643.849178,-8353.685965,11.208792,2.797231e-12,5.594462e-12
1,Gamma,Control MO vs K4K8MO,late,13,13,8696.647077,2678.969673,-6017.677404,10.042294,1.792669e-08,1.792669e-08


In [23]:
qpcrdata=pd.read_excel("data/Fig1/Sup1/Cqresults.xlsx").drop(columns=["Unnamed: 6","Unnamed: 7", "Avg ddcq"], errors="ignore")
qpcrdata.head()

,Sampleid,1,2,3,4
0,K8 1k,6.991887,3.527881,3.492354,5.244302
1,K8 sph,-3.053235,2.031405,1.999770,5.495389
2,K8 sh,-4.525871,-4.613318,-3.961743,-5.231413
3,K8 bud,-5.215579,-4.358182,-5.696946,-4.669629
4,K18 1k,6.841673,3.293707,3.172549,6.914059


In [ ]:
qpcr_long = qpcrdata.melt(id_vars=["Sampleid"], var_name="Replicate", value_name="Cq").copy()
qpcr_long[["gene", "stage"]] = qpcr_long["Sampleid"].str.extract(r"^(K\d+)\s+(.+)$")

for col in ["gene", "stage", "Replicate"]:
    qpcr_long[col] = qpcr_long[col].astype("category")

display(qpcr_long.head())

# Two-way ANOVA: gene effect, stage effect, and gene-by-stage interaction
qpcr_model = ols("Cq ~ C(gene) * C(stage)", data=qpcr_long).fit()
qpcr_anova = sm.stats.anova_lm(qpcr_model, typ=2)
print("Two-way ANOVA for qPCR data")
display(qpcr_anova)

# Optional: one-way ANOVA within each gene if you want stage differences separately
rows = []
for gene, sub in qpcr_long.groupby("gene"):
    gene_model = ols("Cq ~ C(stage)", data=sub).fit()
    gene_anova = sm.stats.anova_lm(gene_model, typ=2)
    gene_anova["gene"] = gene
    rows.append(gene_anova.reset_index().rename(columns={"index": "term"}))

qpcr_stage_anova_by_gene = pd.concat(rows, ignore_index=True)
print("\nOne-way ANOVA by gene")
display(qpcr_stage_anova_by_gene)

,Sampleid,Replicate,Cq,gene,stage
0,K8 1k,1,6.991887,K8,1k
1,K8 sph,1,-3.053235,K8,sph
2,K8 sh,1,-4.525871,K8,sh
3,K8 bud,1,-5.215579,K8,bud
4,K18 1k,1,6.841673,K18,1k


Two-way ANOVA for qPCR data


,sum_sq,df,F,PR(>F)
C(gene),71.297877,3.0,4.709585,5.826839e-03
C(stage),1370.991060,3.0,90.560885,8.953314e-20
C(gene):C(stage),13.207801,9.0,0.290814,9.740404e-01
Residual,242.222202,48.0,NaN,NaN



One-way ANOVA by gene


,term,sum_sq,df,F,PR(>F),gene
0,C(stage),292.801411,3.0,24.603280,0.000021,K18
1,Residual,47.603639,12.0,NaN,NaN,K18
2,C(stage),393.908148,3.0,17.282670,0.000118,K4
3,Residual,91.168356,12.0,NaN,NaN,K4
4,C(stage),420.720690,3.0,30.004968,0.000007,K5
5,Residual,56.086803,12.0,NaN,NaN,K5
6,C(stage),276.768611,3.0,23.374047,0.000027,K8
7,Residual,47.363404,12.0,NaN,NaN,K8


In [35]:
myptcqdata = pd.read_excel("data/Fig2/Sup2/2_i/Myptcq.xlsx").drop(columns=["Avg", "std"], errors="ignore")
myptcqdatalong = myptcqdata.melt(id_vars=["Sampleid"], var_name="Replicate", value_name="Cq").copy()

# Normalize inconsistent spacing and fix merged token like "budwt".
myptcqdatalong["Sampleid_clean"] = (
    myptcqdatalong["Sampleid"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.replace("budwt", "bud wt", regex=False)
    .str.strip()
)

myptcqdatalong[["gene", "stage", "treatment"]] = myptcqdatalong["Sampleid_clean"].str.extract(
    r"^(K\d+)\s+(sh|bud)\s+(wt|mypt)$"
)

bad_rows = myptcqdatalong[myptcqdatalong[["gene", "stage", "treatment"]].isna().any(axis=1)]
if not bad_rows.empty:
    print("Unparsed Sampleid values:")
    display(bad_rows[["Sampleid", "Sampleid_clean"]].drop_duplicates())

for col in ["gene", "stage", "treatment", "Replicate"]:
    myptcqdatalong[col] = myptcqdatalong[col].astype("category")

display(myptcqdatalong.head())

,Sampleid,Replicate,Cq,Sampleid_clean,gene,stage,treatment
0,K8 sh wt,1,55.055540,K8 sh wt,K8,sh,wt
1,K8 sh mypt,1,41.351802,K8 sh mypt,K8,sh,mypt
2,K8 bud wt,1,33.118853,K8 bud wt,K8,bud,wt
3,K8 bud mypt,1,40.823415,K8 bud mypt,K8,bud,mypt
4,K18 sh wt,1,45.910884,K18 sh wt,K18,sh,wt


In [39]:
myptqpcrmodel = ols("Cq ~ C(gene) * C(stage) * C(treatment)", data=myptcqdatalong).fit()
myptqpcranova = sm.stats.anova_lm(myptqpcrmodel, typ=2)
print("Three-way ANOVA for Mypt qPCR data")
display(myptqpcranova)

Three-way ANOVA for Mypt qPCR data


/Users/snaik/2026-Keratinepithlialspreadingcoordinate-Data/.pixi/envs/default/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 1
  warnings.warn('covariance of constraints does not have full '


,sum_sq,df,F,PR(>F)
C(gene),3631.988132,3.0,23.369623,8.828480e-08
C(stage),222.839818,1.0,4.301514,4.738929e-02
C(treatment),0.013813,1.0,0.000267,9.870876e-01
C(gene):C(stage),57.271087,3.0,0.368504,7.762864e-01
C(gene):C(treatment),1.372528,3.0,0.008831,9.257980e-01
C(stage):C(treatment),49.401367,1.0,0.953603,3.371660e-01
C(gene):C(stage):C(treatment),188.190645,3.0,1.210892,3.240069e-01
Residual,1450.539288,28.0,NaN,NaN


In [40]:
rows = []
for gene, sub in myptcqdatalong.groupby("gene"):
    gene_model = ols("Cq ~ C(stage) * C(treatment)", data=sub).fit()
    gene_anova = sm.stats.anova_lm(gene_model, typ=2)
    gene_anova["gene"] = gene
    rows.append(gene_anova.reset_index().rename(columns={"index": "term"}))

mypt_anova_by_gene = pd.concat(rows, ignore_index=True)
print("\nTwo-way ANOVA by gene")
display(mypt_anova_by_gene)


Two-way ANOVA by gene


,term,sum_sq,df,F,PR(>F),gene
0,C(stage),7.114998e+01,1.0,8.116187e-01,0.393954,K18
1,C(treatment),2.098686e+01,1.0,2.394003e-01,0.637772,K18
2,C(stage):C(treatment),1.404766e+02,1.0,1.602439e+00,0.241176,K18
3,Residual,7.013143e+02,8.0,NaN,NaN,K18
4,C(stage),6.740704e+01,1.0,5.162758e+00,0.063478,K4
5,C(treatment),-2.562470e-16,1.0,-1.962616e-17,1.000000,K4
6,C(stage):C(treatment),2.724399e+00,1.0,2.086638e-01,0.663887,K4
7,Residual,7.833841e+01,6.0,NaN,NaN,K4
8,C(stage),1.285956e+02,1.0,7.613460e+00,0.032887,K5
9,C(treatment),2.121580e-14,1.0,1.256074e-15,1.000000,K5


In [5]:
pipdiamdata=pd.read_csv("data/Fig4/Sup4/4_E/PipDiameterViscosity.csv")
rows=[]
for stage, sub in pipdiamdata.groupby("Stage"):
    diammodel=ols("eta ~ C(PipDimension)", data=sub).fit()
    diam_anova=sm.stats.anova_lm(diammodel, typ=2)
    diam_anova["Stage"]=stage
    rows.append(diam_anova.reset_index().rename(columns={"index": "term"}))
pipdiam_anova_by_stage=pd.concat(rows, ignore_index=True)
print("ANOVA for pipette diameter effect on viscosity, by stage")
display(pipdiam_anova_by_stage)

ANOVA for pipette diameter effect on viscosity, by stage


,term,sum_sq,df,F,PR(>F),Stage
0,C(PipDimension),1.098141e+06,2.0,0.342454,0.712378,early
1,Residual,5.611686e+07,35.0,NaN,NaN,early
2,C(PipDimension),1.252804e+07,2.0,0.986914,0.388632,late
3,Residual,1.396357e+08,22.0,NaN,NaN,late


In [17]:
controlhdata=pd.read_csv("data/Fig7/Sup/7E/CHydrodynamicLLength.csv")
YSLmohdata=pd.read_csv("data/Fig7/Sup/7E/YSLHydrodynamicLLength.csv")
combined_hdata = pd.concat(
    [
        controlhdata.rename(columns={controlhdata.columns[0]: "hlength"}).assign(Group="Control"),
        YSLmohdata.rename(columns={YSLmohdata.columns[0]: "hlength"}).assign(Group="YSL MO"),
    ],
    ignore_index=True,
)[["Group", "hlength"]]
combined_hdata["Group"] = combined_hdata["Group"].astype("category")
outpath="Figure/SupFigure7/7D"
combined_hdata.to_csv(os.path.join(outpath, "HydrodynamicLengthData.csv"), index=False)
control=controlhdata
ysl_mo=YSLmohdata
control_df = controlhdata.rename(columns={controlhdata.columns[0]: "hlength"})
ysl_mo_df = YSLmohdata.rename(columns={YSLmohdata.columns[0]: "hlength"})

control = control_df["hlength"].dropna()
ysl_mo = ysl_mo_df["hlength"].dropna()
control_df["hlength"] =1/ control_df["hlength"]
ysl_mo_df["hlength"] = 1/ ysl_mo_df["hlength"]
t_stat, t_p = ttest_ind(control, ysl_mo, equal_var=False)
u_stat, u_p = mannwhitneyu(control, ysl_mo, alternative="two-sided")

print(f"Control mean ± SD: {1/control.mean():.4f} ± {control.std(ddof=1):.4f}")
print(f"YSL MO mean ± SD: {1/ysl_mo.mean():.4f} ± {ysl_mo.std(ddof=1):.4f}")
print(f"Control median: {1/control.median():.4f}")
print(f"YSL MO median: {1/ysl_mo.median():.4f}")
print(f"n Control = {len(control)}")
print(f"n YSL MO = {len(ysl_mo)}")
print(f"Welch t-test: t = {t_stat:.4f}, p = {t_p:.4g}")
print(f"Mann-Whitney U test: U = {u_stat:.4f}, p = {u_p:.4g}")

Control mean ± SD: 5.0680 ± 0.0439
YSL MO mean ± SD: 3.9079 ± 0.0349
Control median: 5.0161
YSL MO median: 4.0125
n Control = 6
n YSL MO = 12
Welch t-test: t = -2.8490, p = 0.02079
Mann-Whitney U test: U = 9.0000, p = 0.009696


In [21]:
maxveldata=pd.read_csv("data/Fig7/Sup/7D/MaxVelocityData.csv")
control_maxvel = maxveldata.loc[maxveldata["Label"] == "Control MO", "MaxVelocity"].dropna()
ysl_mo_maxvel = maxveldata.loc[maxveldata["Label"] == "Keratin 4/8 MO", "MaxVelocity"].dropna()
t_stat_vel, t_p_vel = ttest_ind(control_maxvel, ysl_mo_maxvel, equal_var=False)
u_stat_vel, u_p_vel = mannwhitneyu(control_maxvel, ysl_mo_maxvel, alternative="two-sided")  
print(f"Control Max Velocity mean ± SD: {control_maxvel.mean():.4f} ± {control_maxvel.std(ddof=1):.4f}")
print(f"YSL MO Max Velocity mean ± SD: {ysl_mo_maxvel.mean():.4f} ± {ysl_mo_maxvel.std(ddof=1):.4f}")
print(f"Control Max Velocity median: {control_maxvel.median():.4f}")
print(f"YSL MO Max Velocity median: {ysl_mo_maxvel.median():.4f}")
print(f"n Control Max Velocity = {len(control_maxvel    )}")
print(f"n YSL MO Max Velocity = {len(ysl_mo_maxvel)}")
print(f"Welch t-test for Max Velocity: t = {t_stat_vel:.4f}, p = {t_p_vel:.4g}")
print(f"Mann-Whitney U test for Max Velocity: U = {u_stat_vel:.4f}, p = {u_p_vel:.4g}")


Control Max Velocity mean ± SD: 0.6900 ± 0.1821
YSL MO Max Velocity mean ± SD: 0.4921 ± 0.0816
Control Max Velocity median: 0.6318
YSL MO Max Velocity median: 0.4965
n Control Max Velocity = 7
n YSL MO Max Velocity = 7
Welch t-test for Max Velocity: t = 2.6246, p = 0.02946
Mann-Whitney U test for Max Velocity: U = 44.0000, p = 0.01107


In [4]:
Yslalgindata = pd.read_csv("data/Fig7/7F/YSLalignment.csv")
controldata=Yslalgindata["Control MO"]
yslmodata=Yslalgindata["Keratin YSL MO"]
t_stat_align, t_p_align = ttest_ind(controldata, yslmodata, equal_var=False)
u_stat_align, u_p_align = mannwhitneyu(controldata, yslmodata, alternative="two-sided")  
print(f"Control MO Alignment mean ± SD: {controldata.mean():.4f} ± {controldata.std(ddof=1):.4f}")
print(f"YSL MO Alignment mean ± SD: {yslmodata.mean():.4f} ± {yslmodata.std(ddof=1):.4f}")  
print(f"Control MO Alignment median: {controldata.median():.4f}")
print(f"YSL MO Alignment median: {yslmodata.median():.4f}")
print(f"n Control MO Alignment = {len(controldata)}")
print(f"n YSL MO Alignment = {len(yslmodata)}")
print(f"Welch t-test for Alignment: t = {t_stat_align:.4f}, p = {t_p_align:.4g}")
print(f"Mann-Whitney U test for Alignment: U = {u_stat_align:.4f}, p = {u_p_align:.4g}")

Control MO Alignment mean ± SD: 4.0170 ± 1.3994
YSL MO Alignment mean ± SD: 0.8713 ± 0.5186
Control MO Alignment median: 4.1253
YSL MO Alignment median: 0.7716
n Control MO Alignment = 44
n YSL MO Alignment = 44
Welch t-test for Alignment: t = 13.9813, p = 1.102e-19
Mann-Whitney U test for Alignment: U = 1931.0000, p = 9.561e-16


In [11]:
aspdata=pd.read_csv("data/Fig7/7E/AspAnalysisBox_2024_Figure5.csv")
controlpre  = pd.to_numeric(
    aspdata.loc[aspdata["Sampleid"] == "Control pre", "Intensity"],
    errors="coerce"
).dropna()

controlpost = pd.to_numeric(
    aspdata.loc[aspdata["Sampleid"] == "Control post", "Intensity"],
    errors="coerce"
).dropna()
t_stat_asp, t_p_asp = ttest_rel(controlpre, controlpost)
u_stat_asp, u_p_asp = wilcoxon(controlpre, controlpost, alternative="two-sided")  
print(f"Control pre Asp mean ± SD: {controlpre.mean():.4f} ± {controlpre.std(ddof=1):.4f}")
print(f"Control post Asp mean ± SD: {controlpost.mean():.4f} ± {controlpost.std(ddof=1):.4f}")
print(f"Control pre Asp median: {controlpre.median():.4f}")
print(f"Control post Asp median: {controlpost.median():.4f}")
print(f"n Control pre Asp = {len(controlpre)}") 
print(f"n Control post Asp = {len(controlpost)}")
print(f"Welch t-test for Asp: t = {t_stat_asp:.4f}, p = {t_p_asp:.4g}")
print(f"Wilcoxon test for Asp: W = {u_stat_asp:.4f}, p = {u_p_asp:.4g}")
print("__________________________")
mopre=aspdata.loc[aspdata["Sampleid"] == "MO pre", "Intensity"].dropna()
mopost=aspdata.loc[aspdata["Sampleid"] == "MO post", "Intensity"].dropna()
t_stat_asp_mo, t_p_asp_mo = ttest_ind(mopre, mopost)
u_stat_asp_mo, u_p_asp_mo = wilcoxon(mopre, mopost, alternative="two-sided")
print(f"MO pre Asp mean ± SD: {mopre.mean():.4f} ± {mopre.std(ddof=1):.4f}")
print(f"MO post Asp mean ± SD: {mopost.mean():.4f} ± {mopost.std(ddof=1):.4f}")
print(f"MO pre Asp median: {mopre.median():.4f}")
print(f"MO post Asp median: {mopost.median():.4f}")
print(f"n MO pre Asp = {len(mopre)}")
print(f"n MO post Asp = {len(mopost)}")
print(f"Welch t-test for MO Asp: t = {t_stat_asp_mo:.4f}, p = {t_p_asp_mo:.4g}")
print(f"Wilcoxon test for MO Asp: W = {u_stat_asp_mo:.4f}, p = {u_p_asp_mo:.4g}")

Control pre Asp mean ± SD: 1261.8000 ± 517.0708
Control post Asp mean ± SD: 2430.2000 ± 869.0289
Control pre Asp median: 1322.0000
Control post Asp median: 2438.0000
n Control pre Asp = 5
n Control post Asp = 5
Welch t-test for Asp: t = -6.5141, p = 0.002867
Wilcoxon test for Asp: W = 0.0000, p = 0.0625
__________________________
MO pre Asp mean ± SD: 683.9410 ± 272.6071
MO post Asp mean ± SD: 1030.4810 ± 436.1587
MO pre Asp median: 735.6400
MO post Asp median: 1097.1400
n MO pre Asp = 10
n MO post Asp = 10
Welch t-test for MO Asp: t = -2.1306, p = 0.04717
Wilcoxon test for MO Asp: W = 0.0000, p = 0.001953
